# ETF Volatility Seasonality Study

## Uncovering Calendar-Month Patterns in Realized Volatility

Many commodities and sector ETFs exhibit structural seasonality driven by physical demand cycles, weather, planting/harvest, storage economics, or regulatory calendars. This notebook asks whether that seasonality shows up in realized volatility — and whether the pattern is consistent enough to be tradeable.

### How to use this notebook

1. Set `TICKER` in the config cell (e.g. `'UNG'`, `'USO'`, `'XLE'`, `'TAN'`, `'WEAT'`)
2. Choose your control benchmarks (defaults: SPY + IWM)
3. Run all cells

### What this notebook produces

- Monthly RV profiles (absolute and relative to controls)
- Return seasonality (mean, median, win rate)
- Vol ratio seasonality with IQR bands
- Year × Month heatmaps for RV and vol ratio
- Monthly return distributions (violin plots)
- Seasonal consistency score — how often each month lands in the same vol regime
- Rolling seasonal strength — is the pattern fading or strengthening?
- Auto-detected high/low seasons with cumulative return decomposition

### Controls

- **SPY** — broad market seasonality baseline
- **IWM** — small-cap benchmark (swap for a sector-appropriate control if needed)

---

## 1. Configuration

In [ ]:
# ======================================================================
#  CONFIGURATION — change TICKER and controls here
# ======================================================================

TICKER   = 'UNG'          # The ETF to analyze
CONTROLS = ['SPY', 'IWM'] # Benchmark tickers for relative comparison

START_DATE = '2012-01-01'
END_DATE   = '2025-12-31'

# Rolling RV windows
RV_WINDOW  = 20   # ~1 month realized vol
RV_WINDOW2 = 60   # ~3 month realized vol
ANN_FACTOR = 252

# Derived
ALL_TICKERS = [TICKER] + CONTROLS

MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f"Target: {TICKER}")
print(f"Controls: {CONTROLS}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"RV windows: {RV_WINDOW}d, {RV_WINDOW2}d")

## 2. Setup

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# Color palette — ticker gets the primary color, controls get secondary
COLORS = {
    'ticker': '#1B5E20',
    'ctrl1':  '#00C4E7',
    'ctrl2':  '#F6852B',
    'accent': '#182B40',
    'neutral': '#5A5A5A',
    'green':  '#6DF2A1',
    'gold':   '#F7C940',
    'pink':   '#EC3586',
    'high_season': '#4CAF50',
    'low_season':  '#BDBDBD',
}

def ticker_color(t):
    """Map ticker name to a display color."""
    if t == TICKER:
        return COLORS['ticker']
    idx = CONTROLS.index(t) if t in CONTROLS else 0
    return COLORS['ctrl1'] if idx == 0 else COLORS['ctrl2']

## 3. Fetch Price Data

In [ ]:
print(f"Fetching {ALL_TICKERS} from Yahoo Finance...")
raw = yf.download(ALL_TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw[['Close']]
    prices.columns = ALL_TICKERS

prices = prices.dropna(how='all')
print(f"\nFetched {prices.shape[0]} trading days")
print(f"  {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nMissing values per ticker:")
print(prices.isnull().sum().to_string())

## 4. Compute Returns & Realized Volatility

In [ ]:
# Log returns
log_ret = np.log(prices / prices.shift(1))

# Rolling realized vol (annualized %)
rv20 = log_ret.rolling(RV_WINDOW).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)
rv60 = log_ret.rolling(RV_WINDOW2).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)

# Build panel
df = pd.DataFrame(index=log_ret.index)
for t in ALL_TICKERS:
    df[f'{t}_ret'] = log_ret[t]
    df[f'{t}_rv20'] = rv20[t]
    df[f'{t}_rv60'] = rv60[t]

# Calendar features
df['month'] = df.index.month
df['year'] = df.index.year
df['month_name'] = df.index.strftime('%b')

# Vol ratios vs each control
for ctrl in CONTROLS:
    df[f'{TICKER}_vs_{ctrl}_rv20'] = df[f'{TICKER}_rv20'] / df[f'{ctrl}_rv20']
    df[f'{TICKER}_vs_{ctrl}_rv60'] = df[f'{TICKER}_rv60'] / df[f'{ctrl}_rv60']

print(f"Panel: {len(df)} rows")
print(f"\nRV20 ranges (ann %):")
for t in ALL_TICKERS:
    col = f'{t}_rv20'
    print(f"  {t}: [{df[col].min():.1f}, {df[col].max():.1f}]  median = {df[col].median():.1f}")

## 5. Time Series Overview

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# -- Price
ax = axes[0]
ax.plot(prices.index, prices[TICKER], color=COLORS['ticker'], lw=1)
ax.set_ylabel('Price ($)')
ax.set_title(f'{TICKER}: Price')

# -- RV20 comparison
ax = axes[1]
ax.plot(df.index, df[f'{TICKER}_rv20'], color=COLORS['ticker'], lw=0.8, alpha=0.9, label=TICKER)
for i, ctrl in enumerate(CONTROLS):
    c = COLORS['ctrl1'] if i == 0 else COLORS['ctrl2']
    ax.plot(df.index, df[f'{ctrl}_rv20'], color=c, lw=0.8, alpha=0.6, label=ctrl)
ax.set_ylabel('RV20 (ann %)')
ax.set_title(f'20-Day Realized Volatility: {TICKER} vs Controls')
ax.legend()

# -- Vol ratio vs first control
ctrl0 = CONTROLS[0]
ratio_col = f'{TICKER}_vs_{ctrl0}_rv20'
ax = axes[2]
ax.plot(df.index, df[ratio_col], color=COLORS['accent'], lw=0.8, alpha=0.8)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
med_ratio = df[ratio_col].median()
ax.axhline(med_ratio, color=COLORS['ctrl1'], ls='-', lw=1, alpha=0.5,
           label=f'Median = {med_ratio:.2f}')
ax.set_ylabel(f'{TICKER} RV20 / {ctrl0} RV20')
ax.set_title(f'{TICKER} Vol Ratio vs {ctrl0}')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Monthly Seasonality — Realized Vol by Calendar Month

The core question: does this ETF's realized vol have a month-of-year pattern that differs from broad equity?

In [ ]:
agg_dict = {'n': (f'{TICKER}_rv20', 'count')}
for t in ALL_TICKERS:
    agg_dict[f'{t}_rv20_med'] = (f'{t}_rv20', 'median')
    agg_dict[f'{t}_rv20_mean'] = (f'{t}_rv20', 'mean')

monthly_rv = df.dropna(subset=[f'{TICKER}_rv20']).groupby('month').agg(**agg_dict).round(2)

print(f"{TICKER} Monthly Median RV20 (ann %):\n")
print(monthly_rv.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

months = range(1, 13)
x = np.arange(12)
n_bars = len(ALL_TICKERS)
w = 0.7 / n_bars

# -- Panel 1: Median RV20 by month
ax = axes[0]
for i, t in enumerate(ALL_TICKERS):
    vals = [monthly_rv.loc[m, f'{t}_rv20_med'] for m in months]
    offset = (i - (n_bars - 1) / 2) * w
    ax.bar(x + offset, vals, w, label=t, color=ticker_color(t), alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title(f'Median 20-Day Realized Vol by Calendar Month')
ax.legend()

# -- Panel 2: Excess vol over first control
ax = axes[1]
ctrl0 = CONTROLS[0]
ticker_vals = [monthly_rv.loc[m, f'{TICKER}_rv20_med'] for m in months]
ctrl_vals = [monthly_rv.loc[m, f'{ctrl0}_rv20_med'] for m in months]
excess = [ticker_vals[i] - ctrl_vals[i] for i in range(12)]
bar_colors = [COLORS['ticker'] if e > 0 else COLORS['low_season'] for e in excess]
ax.bar(x, excess, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(0, color='black', lw=1)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel(f'{TICKER} Median RV20 \u2212 {ctrl0} Median RV20 (pp)')
ax.set_title(f'{TICKER} Excess Volatility Over {ctrl0} by Month')

plt.tight_layout()
plt.show()

## 7. Monthly Seasonality — Returns

Does the ETF show return seasonality alongside vol seasonality?

In [ ]:
# Monthly simple returns (sum of daily log returns per calendar month)
monthly_ret = df.groupby([df.index.year, df.index.month]).agg(
    **{f'{t}_ret': (f'{t}_ret', 'sum') for t in ALL_TICKERS}
)
monthly_ret.index.names = ['year', 'month']
monthly_ret = monthly_ret.reset_index()
# Convert log returns to simple returns for display
for t in ALL_TICKERS:
    monthly_ret[f'{t}_ret'] = (np.exp(monthly_ret[f'{t}_ret']) - 1) * 100  # percent

ret_agg = {}
for t in ALL_TICKERS:
    ret_agg[f'{t}_mean'] = (f'{t}_ret', 'mean')
    ret_agg[f'{t}_med'] = (f'{t}_ret', 'median')
ret_agg[f'{TICKER}_std'] = (f'{TICKER}_ret', 'std')
ret_agg[f'{TICKER}_pct_pos'] = (f'{TICKER}_ret', lambda x: (x > 0).mean() * 100)
ret_agg['n'] = (f'{TICKER}_ret', 'count')

monthly_ret_season = monthly_ret.groupby('month').agg(**ret_agg).round(2)

print(f"{TICKER} Monthly Return Seasonality (%):\n")
print(monthly_ret_season.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x = np.arange(12)
n_bars = len(ALL_TICKERS)
w = 0.7 / n_bars

# -- Panel 1: Mean monthly returns
ax = axes[0]
for i, t in enumerate(ALL_TICKERS):
    vals = [monthly_ret_season.loc[m, f'{t}_mean'] for m in range(1, 13)]
    offset = (i - (n_bars - 1) / 2) * w
    ax.bar(x + offset, vals, w, label=t, color=ticker_color(t), alpha=0.8)
ax.axhline(0, color='black', lw=1)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Mean Monthly Return (%)')
ax.set_title('Mean Monthly Return by Calendar Month')
ax.legend()

# -- Panel 2: % positive months (win rate)
ax = axes[1]
pct_pos = [monthly_ret_season.loc[m, f'{TICKER}_pct_pos'] for m in range(1, 13)]
bar_colors = [COLORS['ticker'] if p > 50 else COLORS['pink'] for p in pct_pos]
ax.bar(x, pct_pos, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='50%')

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('% of Months Positive')
ax.set_title(f'{TICKER}: Win Rate by Calendar Month')
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.show()

## 8. Vol Ratio Seasonality — Is the Relative Vol Itself Seasonal?

If the vol ratio (ETF RV20 / SPY RV20) is systematically higher in certain months, the ETF has idiosyncratic seasonal vol beyond what the broad market contributes.

In [ ]:
ctrl0 = CONTROLS[0]
ratio_col = f'{TICKER}_vs_{ctrl0}_rv20'

vol_ratio_monthly = df.dropna(subset=[ratio_col]).groupby('month').agg(
    ratio_mean=(ratio_col, 'mean'),
    ratio_med=(ratio_col, 'median'),
    ratio_p25=(ratio_col, lambda x: np.percentile(x, 25)),
    ratio_p75=(ratio_col, lambda x: np.percentile(x, 75)),
    n=(ratio_col, 'count'),
).round(3)

print(f"{TICKER}/{ctrl0} Vol Ratio by Calendar Month:\n")
print(vol_ratio_monthly.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(12)
meds = [vol_ratio_monthly.loc[m, 'ratio_med'] for m in range(1, 13)]
p25 = [vol_ratio_monthly.loc[m, 'ratio_p25'] for m in range(1, 13)]
p75 = [vol_ratio_monthly.loc[m, 'ratio_p75'] for m in range(1, 13)]

ax.fill_between(x, p25, p75, alpha=0.2, color=COLORS['ticker'], label='p25\u2013p75')
ax.plot(x, meds, 'o-', color=COLORS['ticker'], lw=2, markersize=8, label='Median', zorder=5)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5, label='Ratio = 1.0')

# Shade months where median ratio is above overall median (auto-detected high season)
overall_med = df[ratio_col].median()
for m_idx in range(12):
    if meds[m_idx] > overall_med:
        ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['high_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel(f'{TICKER} RV20 / {ctrl0} RV20')
ax.set_title(f'{TICKER}/{ctrl0} Vol Ratio Seasonality by Calendar Month\n(Shading = months above overall median ratio)')
ax.legend()

plt.tight_layout()
plt.show()

## 9. Year-Over-Year Heatmap — Monthly RV20

Each cell shows the ETF's median RV20 for that year × month. Reveals whether the seasonal pattern is stable across years or driven by outlier years.

In [ ]:
ym_rv = df.dropna(subset=[f'{TICKER}_rv20']).groupby(['year', 'month'])[f'{TICKER}_rv20'].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_rv) * 0.5)))

vals = ym_rv.values
im = ax.imshow(vals, cmap='YlOrRd', aspect='auto',
               vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if v > np.nanpercentile(vals, 70) else 'black'
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_rv)))
ax.set_yticklabels(ym_rv.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title(f'{TICKER} Median RV20 (ann %) by Year \u00d7 Month')
plt.colorbar(im, shrink=0.8, label='RV20 (ann %)')

plt.tight_layout()
plt.show()

## 10. Year-Over-Year Heatmap — Vol Ratio

Same heatmap but for the vol ratio vs the primary control. This controls for market-wide vol regimes.

In [ ]:
ctrl0 = CONTROLS[0]
ratio_col = f'{TICKER}_vs_{ctrl0}_rv20'

ym_ratio = df.dropna(subset=[ratio_col]).groupby(['year', 'month'])[ratio_col].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_ratio) * 0.5)))

vals = ym_ratio.values
vmax = max(abs(np.nanpercentile(vals, 5) - 1), abs(np.nanpercentile(vals, 95) - 1))
im = ax.imshow(vals, cmap='RdYlGn_r', aspect='auto',
               vmin=1 - vmax, vmax=1 + vmax)

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if abs(v - 1) > vmax * 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_ratio)))
ax.set_yticklabels(ym_ratio.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title(f'{TICKER}/{ctrl0} Vol Ratio by Year \u00d7 Month\n(>1 = {TICKER} more volatile than {ctrl0})')
plt.colorbar(im, shrink=0.8, label=f'{TICKER} RV20 / {ctrl0} RV20')

plt.tight_layout()
plt.show()

## 11. Monthly Return Distributions — Violin Plots

Violin plots show the full distribution of monthly returns for each calendar month. This reveals skew — months where returns are not just volatile but directionally biased.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# -- Panel 1: Monthly return distributions
ax = axes[0]
month_data = [monthly_ret.loc[monthly_ret.month == m, f'{TICKER}_ret'].values for m in range(1, 13)]
parts = ax.violinplot(month_data, positions=range(12), showmedians=True, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor(COLORS['ticker'])
    pc.set_alpha(0.4)
parts['cmedians'].set_color(COLORS['ticker'])
parts['cmedians'].set_linewidth(2)

# Overlay individual data points
for m in range(12):
    y = month_data[m]
    jitter = np.random.normal(0, 0.05, size=len(y))
    ax.scatter(m + jitter, y, alpha=0.3, s=12, color=COLORS['ticker'], zorder=3)

ax.axhline(0, color='black', lw=1)
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Monthly Return (%)')
ax.set_title(f'{TICKER}: Monthly Return Distributions')

# -- Panel 2: Monthly RV20 distributions (box plots)
ax = axes[1]
rv_data = [df.dropna(subset=[f'{TICKER}_rv20']).loc[df.month == m, f'{TICKER}_rv20'].values for m in range(1, 13)]
bp = ax.boxplot(rv_data, positions=range(12), widths=0.5, patch_artist=True,
                showfliers=False, medianprops={'color': 'black', 'lw': 2})

# Color boxes by above/below median vol
overall_med_rv = df[f'{TICKER}_rv20'].median()
for i, patch in enumerate(bp['boxes']):
    month_med = np.median(rv_data[i])
    patch.set_facecolor(COLORS['ticker'] if month_med > overall_med_rv else COLORS['low_season'])
    patch.set_alpha(0.6)

ax.axhline(overall_med_rv, color=COLORS['pink'], ls='--', lw=1, alpha=0.7,
           label=f'Overall median = {overall_med_rv:.1f}%')
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('RV20 (ann %)')
ax.set_title(f'{TICKER}: RV20 Distribution by Calendar Month')
ax.legend()

plt.tight_layout()
plt.show()

## 12. Seasonal Consistency Score

A month with high *average* vol might be driven by one crazy year. The consistency score measures how reliably each month lands in the top or bottom half of that year's vol distribution. A score near 100% means that month is *always* relatively high (or low) vol — the pattern repeats. A score near 50% means it's a coin flip.

In [ ]:
# For each year, rank months by median RV20. Then for each month, count how often
# it falls in the top 6 or bottom 6 months.
ym_rv_full = df.dropna(subset=[f'{TICKER}_rv20']).groupby(['year', 'month'])[f'{TICKER}_rv20'].median().unstack()

# Only use years with all 12 months of data
complete_years = ym_rv_full.dropna(axis=0)
n_years = len(complete_years)

# For each year, rank months (1=lowest vol, 12=highest vol)
ranks = complete_years.rank(axis=1)

# Median rank per month across years
median_rank = ranks.median()

# Consistency: for months with median rank > 6 (high vol), count % of years in top 6
#              for months with median rank <= 6 (low vol), count % of years in bottom 6
consistency = pd.Series(index=range(1, 13), dtype=float)
regime = pd.Series(index=range(1, 13), dtype=str)
for m in range(1, 13):
    if median_rank[m] > 6:
        consistency[m] = (ranks[m] > 6).mean() * 100
        regime[m] = 'High Vol'
    else:
        consistency[m] = (ranks[m] <= 6).mean() * 100
        regime[m] = 'Low Vol'

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# -- Panel 1: Consistency score
ax = axes[0]
bar_colors = [COLORS['ticker'] if r == 'High Vol' else COLORS['low_season'] for r in regime.values]
bars = ax.bar(range(12), consistency.values, 0.6, color=bar_colors, alpha=0.8,
              edgecolor='white', linewidth=0.5)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='Random (50%)')

for i, (score, reg) in enumerate(zip(consistency.values, regime.values)):
    ax.text(i, score + 1.5, f'{score:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(12))
ax.set_xticklabels([f'{MONTH_NAMES[i]}\n({regime.iloc[i]})' for i in range(12)], fontsize=9)
ax.set_ylabel('Consistency (%)')
ax.set_title(f'{TICKER}: Seasonal Consistency Score\n(% of years month falls in expected vol regime, n={n_years} years)')
ax.set_ylim(0, 110)
ax.legend()

# -- Panel 2: Median vol rank by month (1=lowest, 12=highest)
ax = axes[1]
rank_vals = median_rank.values
bar_colors2 = [COLORS['ticker'] if r > 6 else COLORS['low_season'] for r in rank_vals]
ax.bar(range(12), rank_vals, 0.6, color=bar_colors2, alpha=0.8)
ax.axhline(6.5, color='black', ls='--', lw=1, alpha=0.5)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Median Vol Rank (1=lowest, 12=highest)')
ax.set_title(f'{TICKER}: Median Volatility Rank by Calendar Month')
ax.set_ylim(0, 13)

plt.tight_layout()
plt.show()

print(f"\nUsing {n_years} complete years: {sorted(complete_years.index.tolist())}")
print(f"\nMost consistent HIGH vol months: ", end='')
high_months = consistency[regime == 'High Vol'].sort_values(ascending=False)
print(', '.join([f"{MONTH_NAMES[m-1]} ({high_months[m]:.0f}%)" for m in high_months.index[:3]]))
print(f"Most consistent LOW vol months:  ", end='')
low_months = consistency[regime == 'Low Vol'].sort_values(ascending=False)
print(', '.join([f"{MONTH_NAMES[m-1]} ({low_months[m]:.0f}%)" for m in low_months.index[:3]]))

## 13. Rolling Seasonal Strength — Is the Pattern Fading?

We compute the ratio of high-season median RV to low-season median RV using a 3-year rolling window. If this ratio is declining, the seasonal edge may be arbitraged away. If it's increasing, the structural driver is strengthening.

In [ ]:
# Identify high and low season months from the consistency analysis
high_months_set = set(regime[regime == 'High Vol'].index)
low_months_set = set(regime[regime == 'Low Vol'].index)

high_labels = ', '.join([MONTH_NAMES[m-1] for m in sorted(high_months_set)])
low_labels = ', '.join([MONTH_NAMES[m-1] for m in sorted(low_months_set)])
print(f"Auto-detected seasons:")
print(f"  High vol months: {high_labels}")
print(f"  Low vol months:  {low_labels}")

# For each year, compute median RV in high vs low season
yearly_seasonal = pd.DataFrame(index=complete_years.index)
yearly_seasonal['high_season_rv'] = complete_years[list(high_months_set)].median(axis=1)
yearly_seasonal['low_season_rv'] = complete_years[list(low_months_set)].median(axis=1)
yearly_seasonal['seasonal_ratio'] = yearly_seasonal['high_season_rv'] / yearly_seasonal['low_season_rv']

# Also compute rolling 3-year version for smoothing
rolling_window = min(3, n_years - 1)  # use 3 years or less if not enough data
yearly_seasonal['rolling_ratio'] = yearly_seasonal['seasonal_ratio'].rolling(rolling_window, min_periods=2).mean()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# -- Panel 1: High vs low season RV by year
ax = axes[0]
years = yearly_seasonal.index
x = np.arange(len(years))
w = 0.35
ax.bar(x - w/2, yearly_seasonal['high_season_rv'], w, label=f'High season ({high_labels})',
       color=COLORS['ticker'], alpha=0.8)
ax.bar(x + w/2, yearly_seasonal['low_season_rv'], w, label=f'Low season ({low_labels})',
       color=COLORS['low_season'], alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title(f'{TICKER}: High vs Low Season Median RV by Year')
ax.legend(fontsize=9)

# -- Panel 2: Seasonal ratio over time
ax = axes[1]
ax.bar(x, yearly_seasonal['seasonal_ratio'], 0.6, color=COLORS['ticker'], alpha=0.4, label='Annual ratio')
ax.plot(x, yearly_seasonal['rolling_ratio'], 'o-', color=COLORS['accent'], lw=2,
        markersize=6, label=f'{rolling_window}-yr rolling avg', zorder=5)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5, label='No seasonality')

ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45)
ax.set_ylabel('High Season RV / Low Season RV')
ax.set_title(f'{TICKER}: Seasonal Strength Over Time\n(>1 = high season is more volatile)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nOverall seasonal ratio: {yearly_seasonal['seasonal_ratio'].median():.2f}x")
recent = yearly_seasonal['seasonal_ratio'].iloc[-3:]
print(f"Recent 3-year median:   {recent.median():.2f}x")

## 14. Seasonal Radar — Vol Profile at a Glance

A radar (polar) chart showing the ETF's median RV by month alongside controls. The shape of the polygon immediately reveals which months are hot and which are quiet.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw={'projection': 'polar'})

angles = np.linspace(0, 2 * np.pi, 12, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

for t in ALL_TICKERS:
    vals = [monthly_rv.loc[m, f'{t}_rv20_med'] for m in range(1, 13)]
    vals += vals[:1]  # close
    c = ticker_color(t)
    lw = 2.5 if t == TICKER else 1.2
    alpha = 0.9 if t == TICKER else 0.5
    ax.plot(angles, vals, 'o-', color=c, lw=lw, alpha=alpha, label=t, markersize=4)
    if t == TICKER:
        ax.fill(angles, vals, color=c, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(MONTH_NAMES, fontsize=10)
ax.set_title(f'{TICKER} Seasonal Vol Profile\n(Median RV20, ann %)', y=1.08, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

## 15. Cumulative Return — High Season vs Low Season

Using the auto-detected high and low vol seasons, we decompose cumulative returns into what you'd earn holding only during high-vol months vs only during low-vol months.

In [ ]:
df_valid = df.dropna(subset=[f'{TICKER}_ret']).copy()

df_valid['is_high'] = df_valid['month'].isin(high_months_set)
df_valid['high_ret'] = np.where(df_valid['is_high'], df_valid[f'{TICKER}_ret'], 0)
df_valid['low_ret'] = np.where(~df_valid['is_high'], df_valid[f'{TICKER}_ret'], 0)
df_valid['all_ret'] = df_valid[f'{TICKER}_ret']

df_valid['cum_high'] = df_valid['high_ret'].cumsum()
df_valid['cum_low'] = df_valid['low_ret'].cumsum()
df_valid['cum_all'] = df_valid['all_ret'].cumsum()
df_valid['cum_ctrl'] = df_valid[f'{CONTROLS[0]}_ret'].cumsum()

fig, ax = plt.subplots(figsize=(16, 8))
ax.plot(df_valid.index, (np.exp(df_valid['cum_high']) - 1) * 100,
        color=COLORS['ticker'], lw=1.5, label=f'{TICKER}: High Vol Months Only ({high_labels})')
ax.plot(df_valid.index, (np.exp(df_valid['cum_low']) - 1) * 100,
        color=COLORS['low_season'], lw=1.5, label=f'{TICKER}: Low Vol Months Only ({low_labels})')
ax.plot(df_valid.index, (np.exp(df_valid['cum_all']) - 1) * 100,
        color=COLORS['accent'], lw=1, alpha=0.5, label=f'{TICKER}: All Months')
ax.plot(df_valid.index, (np.exp(df_valid['cum_ctrl']) - 1) * 100,
        color=COLORS['ctrl1'], lw=1, alpha=0.5, label=f'{CONTROLS[0]}: All Months')

ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Cumulative Return (%)')
ax.set_title(f'{TICKER} Cumulative Returns: High Vol Season vs Low Vol Season')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

years_span = (df_valid.index[-1] - df_valid.index[0]).days / 365.25
for label, col in [('High Vol Months', 'cum_high'), ('Low Vol Months', 'cum_low'), ('All Months', 'cum_all')]:
    total = df_valid[col].iloc[-1]
    ann_ret = (np.exp(total / years_span) - 1) * 100
    print(f"  {TICKER} {label}: {(np.exp(total)-1)*100:.1f}% total, ~{ann_ret:.1f}% ann")

## 16. Summary Statistics

In [ ]:
print("=" * 70)
print(f"{TICKER} SEASONALITY SUMMARY")
print("=" * 70)

valid = df.dropna(subset=[f'{TICKER}_rv20', f'{CONTROLS[0]}_rv20'])

high = valid[valid.month.isin(high_months_set)]
low = valid[valid.month.isin(low_months_set)]

print(f"\nAuto-detected seasons:")
print(f"  High vol months: {high_labels}")
print(f"  Low vol months:  {low_labels}")

print(f"\n{TICKER} Median RV20:")
print(f"  High season: {high[f'{TICKER}_rv20'].median():.1f}%")
print(f"  Low season:  {low[f'{TICKER}_rv20'].median():.1f}%")
print(f"  Ratio:       {high[f'{TICKER}_rv20'].median() / low[f'{TICKER}_rv20'].median():.2f}x")

ctrl0 = CONTROLS[0]
print(f"\n{ctrl0} Median RV20 (control):")
print(f"  High season: {high[f'{ctrl0}_rv20'].median():.1f}%")
print(f"  Low season:  {low[f'{ctrl0}_rv20'].median():.1f}%")
print(f"  Ratio:       {high[f'{ctrl0}_rv20'].median() / low[f'{ctrl0}_rv20'].median():.2f}x")

ratio_col = f'{TICKER}_vs_{ctrl0}_rv20'
print(f"\n{TICKER}/{ctrl0} Vol Ratio:")
print(f"  High season median: {high[ratio_col].median():.3f}")
print(f"  Low season median:  {low[ratio_col].median():.3f}")

print(f"\n{TICKER} Mean Daily Return (annualized):")
print(f"  High season: {high[f'{TICKER}_ret'].mean() * ANN_FACTOR * 100:.1f}%")
print(f"  Low season:  {low[f'{TICKER}_ret'].mean() * ANN_FACTOR * 100:.1f}%")

print(f"\nSeasonal consistency (from section 12):")
print(f"  Avg high-season consistency: {consistency[regime == 'High Vol'].mean():.0f}%")
print(f"  Avg low-season consistency:  {consistency[regime == 'Low Vol'].mean():.0f}%")

print(f"\n" + "=" * 70)

## 17. Export

In [ ]:
output_path = f'{TICKER.lower()}_seasonality_panel.csv'
export_cols = [c for c in df.columns if '_ret' in c or '_rv20' in c or '_rv60' in c or '_vs_' in c]
export_cols += ['month', 'year']
df[export_cols].to_csv(output_path)
print(f"Exported to {output_path}")
print(f"  {df.shape[0]:,} rows")